# M1 4분류 1단계 Binary Gate 검증

## tl;dr

이 노트북은 `normal / fault / task / activity` 4분류로 바로 들어가기 전에, 1단계 gate인 `normal vs event`가 가능한지 검증한다. `event`는 `fault + task + activity` 전체이며, 기존 `efd_possible/pre_event`는 fault 내부 보조 실험으로만 유지한다.

실행 후 `07_데이터산출물`에 dataset summary, metrics, predictions, threshold metrics, decision matrix, 보고서를 저장한다.

## Context & Methods

- 입력은 16번 taxonomy 산출물을 사용한다.
- label은 `normal=0`, `event=1`로 만든다.
- 비교 dataset은 `pre_event_all`과 `low_overlap_hybrid` 두 개다.
- feature는 `compact13`, `expanded154`를 분리해서 비교한다.
- model은 Dummy, Logistic Regression, RandomForest, ExtraTrees, HistGradientBoosting을 비교한다.
- 검증은 `substation_id` 기준 group CV로 수행한다.

### Key Assumptions

- 회사 제공 normal 35건은 유지한다.
- event는 fault/task/activity 전체다.
- metadata, 날짜, event id, substation, coverage는 학습 feature에서 제외한다.
- threshold 0.5를 기본 판단 기준으로 두고, 0.4/0.6은 참고용으로 산출한다.

In [1]:
from __future__ import annotations

from pathlib import Path
import json
import math
import warnings

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

PROJECT_ROOT = Path.cwd().resolve()
OUTPUT_DIR = PROJECT_ROOT / '07_데이터산출물'

LABEL_INDEX_PATH = OUTPUT_DIR / 'm1_4class_label_candidate_index.csv'
WINDOW_AUDIT_PATH = OUTPUT_DIR / 'm1_4class_window_policy_audit.csv'
FEASIBILITY_PATH = OUTPUT_DIR / 'm1_4class_dataset_feasibility_summary.csv'
FEATURE_SET_PATH = OUTPUT_DIR / 'm1_compact_feature_set_summary.csv'

OUT_DATASET_SUMMARY = OUTPUT_DIR / 'm1_4class_binary_gate_dataset_summary.csv'
OUT_METRICS = OUTPUT_DIR / 'm1_4class_binary_gate_metrics.csv'
OUT_PREDICTIONS = OUTPUT_DIR / 'm1_4class_binary_gate_predictions.csv'
OUT_THRESHOLD_METRICS = OUTPUT_DIR / 'm1_4class_binary_gate_threshold_metrics.csv'
OUT_DECISION_MATRIX = OUTPUT_DIR / 'm1_4class_binary_gate_decision_matrix.csv'
OUT_REPORT = OUTPUT_DIR / '17_M1_4class_binary_gate_검증_보고서.md'

RANDOM_STATE = 42
N_SPLITS = 5
THRESHOLDS = [0.4, 0.5, 0.6]
DEFAULT_THRESHOLD = 0.5

assert LABEL_INDEX_PATH.exists(), LABEL_INDEX_PATH
assert WINDOW_AUDIT_PATH.exists(), WINDOW_AUDIT_PATH
assert FEASIBILITY_PATH.exists(), FEASIBILITY_PATH
assert FEATURE_SET_PATH.exists(), FEATURE_SET_PATH

print('project_root:', PROJECT_ROOT)
print('output_dir:', OUTPUT_DIR)

project_root: C:\3rd_Project\HeatGridAgent
output_dir: C:\3rd_Project\HeatGridAgent\07_데이터산출물


## Data

16번 taxonomy 산출물에서 label 후보, window audit, feature set 정의를 불러온다.

In [2]:
label_index = pd.read_csv(LABEL_INDEX_PATH)
window_audit = pd.read_csv(WINDOW_AUDIT_PATH)
feasibility = pd.read_csv(FEASIBILITY_PATH)
feature_sets = pd.read_csv(FEATURE_SET_PATH)

for frame_name, frame in [('label_index', label_index), ('window_audit', window_audit)]:
    required = {'source_id', 'final_class', 'window_policy', 'coverage_eligible', 'substation_id'}
    missing = required - set(frame.columns)
    assert not missing, f'{frame_name} missing columns: {missing}'

print('label_index:', label_index.shape)
print('window_audit:', window_audit.shape)
print('feature_sets:', feature_sets[['feature_set', 'feature_count']].to_string(index=False))

label_index: (200, 190)
window_audit: (298, 190)
feature_sets:       feature_set  feature_count
           base70             70
      expanded154            154
compact13_overlap             13
   compact20_main             20
  compact27_union             27


In [3]:
def parse_feature_list(feature_set_name: str) -> list[str]:
    row = feature_sets.loc[feature_sets['feature_set'] == feature_set_name]
    assert len(row) == 1, f'feature set not found: {feature_set_name}'
    features = str(row.iloc[0]['features']).split('|')
    features = [feature for feature in features if feature]
    missing = sorted(set(features) - set(label_index.columns))
    assert not missing, f'{feature_set_name} features missing in label index: {missing[:10]}'
    return features

FEATURE_COLUMNS = {
    'compact13': parse_feature_list('compact13_overlap'),
    'expanded154': parse_feature_list('expanded154'),
}

for name, cols in FEATURE_COLUMNS.items():
    print(name, len(cols))
    assert len(cols) in {13, 154}, (name, len(cols))

compact13 13
expanded154 154


## Build Binary Gate Datasets

- `pre_event_all`: 16번 baseline index 그대로 사용한다.
- `low_overlap_hybrid`: normal/fault/activity는 동일하게 두고, task만 `task_post_7d` window로 바꿔 task pre-window overlap 영향을 줄인다.

In [4]:
def as_bool(series: pd.Series) -> pd.Series:
    if series.dtype == bool:
        return series.fillna(False)
    return series.astype(str).str.lower().isin(['true', '1', 'yes'])


def build_pre_event_all() -> pd.DataFrame:
    data = label_index.loc[as_bool(label_index['baseline_included'])].copy()
    data['dataset'] = 'pre_event_all'
    return data


def build_low_overlap_hybrid() -> pd.DataFrame:
    eligible = as_bool(window_audit['coverage_eligible'])
    parts = []
    parts.append(window_audit.loc[(window_audit['final_class'] == 'normal') & (window_audit['window_policy'] == 'normal_event_7d') & eligible].copy())
    parts.append(window_audit.loc[(window_audit['final_class'] == 'fault') & (window_audit['window_policy'] == 'fault_pre_7d') & eligible].copy())
    parts.append(window_audit.loc[(window_audit['final_class'] == 'task') & (window_audit['window_policy'] == 'task_post_7d') & eligible].copy())
    parts.append(window_audit.loc[(window_audit['final_class'] == 'activity') & (window_audit['window_policy'] == 'activity_pre_7d') & eligible].copy())
    data = pd.concat(parts, ignore_index=True)
    data['dataset'] = 'low_overlap_hybrid'
    return data


def finalize_binary_dataset(data: pd.DataFrame) -> pd.DataFrame:
    data = data.copy()
    data['binary_label'] = np.where(data['final_class'].eq('normal'), 0, 1)
    data['binary_label_name'] = np.where(data['binary_label'].eq(0), 'normal', 'event')
    data['event_type_binary'] = np.where(data['final_class'].eq('normal'), 'normal', data['final_class'])
    data['substation_id'] = data['substation_id'].astype(int)
    duplicate_ids = data['source_id'][data['source_id'].duplicated()].unique().tolist()
    assert not duplicate_ids, f'duplicate source_id in dataset: {duplicate_ids[:10]}'
    assert data['binary_label'].nunique() == 2
    return data

DATASETS = {
    'pre_event_all': finalize_binary_dataset(build_pre_event_all()),
    'low_overlap_hybrid': finalize_binary_dataset(build_low_overlap_hybrid()),
}

for name, data in DATASETS.items():
    print('\n==', name, data.shape)
    print(data.groupby(['binary_label_name', 'final_class', 'window_policy']).size().to_string())
    print('substations:', data['substation_id'].nunique())


== pre_event_all (181, 194)
binary_label_name  final_class  window_policy  
event              activity     activity_pre_7d    47
                   fault        fault_pre_7d       62
                   task         task_pre_7d        37
normal             normal       normal_event_7d    35
substations: 33

== low_overlap_hybrid (185, 194)
binary_label_name  final_class  window_policy  
event              activity     activity_pre_7d    47
                   fault        fault_pre_7d       62
                   task         task_post_7d       41
normal             normal       normal_event_7d    35
substations: 33


In [5]:
summary_rows = []
for dataset_name, data in DATASETS.items():
    row = {
        'dataset': dataset_name,
        'rows': len(data),
        'normal_rows': int((data['binary_label'] == 0).sum()),
        'event_rows': int((data['binary_label'] == 1).sum()),
        'fault_rows': int((data['final_class'] == 'fault').sum()),
        'task_rows': int((data['final_class'] == 'task').sum()),
        'activity_rows': int((data['final_class'] == 'activity').sum()),
        'substation_count': int(data['substation_id'].nunique()),
        'coverage_min': float(data['coverage_rate'].min()),
        'coverage_median': float(data['coverage_rate'].median()),
        'overlap_rows': int((pd.to_numeric(data['overlap_disturbance_count'], errors='coerce').fillna(0) > 0).sum()),
        'window_policies': '|'.join(sorted(data['window_policy'].dropna().unique())),
    }
    summary_rows.append(row)

dataset_summary = pd.DataFrame(summary_rows)
dataset_summary.to_csv(OUT_DATASET_SUMMARY, index=False, encoding='utf-8-sig')
dataset_summary

,dataset,rows,normal_rows,event_rows,fault_rows,task_rows,activity_rows,substation_count,coverage_min,coverage_median,overlap_rows,window_policies
0,pre_event_all,181,35,146,62,37,47,33,0.995040,1.0,38,activity_pre_7d|fault_pre_7d|normal_event_7d|t...
1,low_overlap_hybrid,185,35,150,62,41,47,33,0.982143,1.0,15,activity_pre_7d|fault_pre_7d|normal_event_7d|t...


## Models And Group CV

동일한 group split에서 feature set과 model을 나란히 비교한다.

In [6]:
def build_model(model_name: str):
    if model_name == 'dummy_most_frequent':
        return DummyClassifier(strategy='most_frequent')
    if model_name == 'logistic_balanced':
        return Pipeline([
            ('imputer', SimpleImputer(strategy='median', keep_empty_features=True)),
            ('scaler', StandardScaler()),
            ('model', LogisticRegression(class_weight='balanced', max_iter=5000, solver='liblinear', random_state=RANDOM_STATE)),
        ])
    if model_name == 'random_forest_balanced_depth3':
        return Pipeline([
            ('imputer', SimpleImputer(strategy='median', keep_empty_features=True)),
            ('model', RandomForestClassifier(n_estimators=300, max_depth=3, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)),
        ])
    if model_name == 'extra_trees_balanced_depth3':
        return Pipeline([
            ('imputer', SimpleImputer(strategy='median', keep_empty_features=True)),
            ('model', ExtraTreesClassifier(n_estimators=300, max_depth=3, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)),
        ])
    if model_name == 'hist_gradient_boosting':
        return Pipeline([
            ('imputer', SimpleImputer(strategy='median', keep_empty_features=True)),
            ('model', HistGradientBoostingClassifier(max_iter=150, learning_rate=0.05, max_leaf_nodes=15, random_state=RANDOM_STATE)),
        ])
    raise ValueError(model_name)

MODEL_NAMES = [
    'dummy_most_frequent',
    'logistic_balanced',
    'random_forest_balanced_depth3',
    'extra_trees_balanced_depth3',
    'hist_gradient_boosting',
]


def clean_feature_matrix(data: pd.DataFrame, feature_cols: list[str]) -> pd.DataFrame:
    X = data[feature_cols].apply(pd.to_numeric, errors='coerce')
    X = X.replace([np.inf, -np.inf], np.nan)
    return X


def get_event_probability(model, X: pd.DataFrame) -> np.ndarray:
    if hasattr(model, 'predict_proba'):
        proba = model.predict_proba(X)
    elif isinstance(model, Pipeline) and hasattr(model[-1], 'predict_proba'):
        proba = model.predict_proba(X)
    else:
        decision = model.decision_function(X)
        proba_event = 1 / (1 + np.exp(-decision))
        return proba_event
    classes = list(model.classes_) if hasattr(model, 'classes_') else list(model[-1].classes_)
    event_idx = classes.index(1)
    return proba[:, event_idx]


def metric_row(y_true, y_prob, threshold: float) -> dict:
    y_pred = (np.asarray(y_prob) >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    normal_fpr = fp / (fp + tn) if (fp + tn) else np.nan
    event_recall = tp / (tp + fn) if (tp + fn) else np.nan
    try:
        auc = roc_auc_score(y_true, y_prob)
    except ValueError:
        auc = np.nan
    return {
        'threshold': threshold,
        'rows': int(len(y_true)),
        'accuracy': accuracy_score(y_true, y_pred),
        'balanced_accuracy': balanced_accuracy_score(y_true, y_pred),
        'roc_auc': auc,
        'precision_event': precision_score(y_true, y_pred, zero_division=0),
        'recall_event': recall_score(y_true, y_pred, zero_division=0),
        'f1_event': f1_score(y_true, y_pred, zero_division=0),
        'normal_fpr': normal_fpr,
        'tn': int(tn),
        'fp': int(fp),
        'fn': int(fn),
        'tp': int(tp),
    }

print('models:', MODEL_NAMES)

models: ['dummy_most_frequent', 'logistic_balanced', 'random_forest_balanced_depth3', 'extra_trees_balanced_depth3', 'hist_gradient_boosting']


In [7]:
prediction_rows = []
threshold_metric_rows = []
metric_rows = []
fold_overlap_rows = []

for dataset_name, data in DATASETS.items():
    y = data['binary_label'].astype(int).to_numpy()
    groups = data['substation_id'].astype(int).to_numpy()
    splitter = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
    splits = list(splitter.split(data, y, groups))

    for feature_set_name, feature_cols in FEATURE_COLUMNS.items():
        X = clean_feature_matrix(data, feature_cols)

        for fold, (train_idx, test_idx) in enumerate(splits, start=1):
            train_groups = set(groups[train_idx])
            test_groups = set(groups[test_idx])
            overlap = sorted(train_groups & test_groups)
            fold_overlap_rows.append({
                'dataset': dataset_name,
                'feature_set': feature_set_name,
                'fold': fold,
                'train_rows': int(len(train_idx)),
                'test_rows': int(len(test_idx)),
                'train_groups': '|'.join(map(str, sorted(train_groups))),
                'test_groups': '|'.join(map(str, sorted(test_groups))),
                'group_overlap_count': int(len(overlap)),
            })
            assert not overlap, f'group overlap: {dataset_name} {feature_set_name} fold {fold} {overlap}'

            X_train = X.iloc[train_idx]
            X_test = X.iloc[test_idx]
            y_train = y[train_idx]
            y_test = y[test_idx]
            test_data = data.iloc[test_idx].copy()

            for model_name in MODEL_NAMES:
                model = build_model(model_name)
                model.fit(X_train, y_train)
                y_prob = get_event_probability(model, X_test)

                pred_frame = test_data[[
                    'source_id', 'final_class', 'event_type_binary', 'source_event_id',
                    'disturbance_row_id', 'substation_id', 'window_policy', 'window_start',
                    'window_end', 'coverage_rate', 'overlap_disturbance_count', 'hard_normal_tag'
                ]].copy()
                pred_frame['dataset'] = dataset_name
                pred_frame['feature_set'] = feature_set_name
                pred_frame['feature_count'] = len(feature_cols)
                pred_frame['model'] = model_name
                pred_frame['fold'] = fold
                pred_frame['y_true'] = y_test
                pred_frame['probability_event'] = y_prob
                for threshold in THRESHOLDS:
                    pred_frame[f'prediction_t{str(threshold).replace(".", "_")}'] = np.where(y_prob >= threshold, 1, 0)
                prediction_rows.extend(pred_frame.to_dict('records'))

                for threshold in THRESHOLDS:
                    row = metric_row(y_test, y_prob, threshold)
                    row.update({
                        'dataset': dataset_name,
                        'feature_set': feature_set_name,
                        'feature_count': len(feature_cols),
                        'model': model_name,
                        'fold': fold,
                        'metric_scope': 'fold',
                        'train_rows': int(len(train_idx)),
                        'test_rows': int(len(test_idx)),
                        'train_groups': '|'.join(map(str, sorted(train_groups))),
                        'test_groups': '|'.join(map(str, sorted(test_groups))),
                        'group_overlap_count': 0,
                    })
                    threshold_metric_rows.append(row)
                    if threshold == DEFAULT_THRESHOLD:
                        metric_rows.append(row.copy())

predictions = pd.DataFrame(prediction_rows)
threshold_metrics_fold = pd.DataFrame(threshold_metric_rows)
metrics_fold = pd.DataFrame(metric_rows)

print('predictions:', predictions.shape)
print('fold threshold metrics:', threshold_metrics_fold.shape)
print('default threshold fold metrics:', metrics_fold.shape)

predictions: (3660, 22)
fold threshold metrics: (300, 24)
default threshold fold metrics: (100, 24)


In [8]:
aggregate_rows = []
for keys, group in predictions.groupby(['dataset', 'feature_set', 'feature_count', 'model'], dropna=False):
    dataset_name, feature_set_name, feature_count, model_name = keys
    y_true = group['y_true'].astype(int).to_numpy()
    y_prob = group['probability_event'].astype(float).to_numpy()
    for threshold in THRESHOLDS:
        row = metric_row(y_true, y_prob, threshold)
        row.update({
            'dataset': dataset_name,
            'feature_set': feature_set_name,
            'feature_count': int(feature_count),
            'model': model_name,
            'fold': 'all',
            'metric_scope': 'aggregate',
            'train_rows': np.nan,
            'test_rows': int(len(group)),
            'train_groups': np.nan,
            'test_groups': np.nan,
            'group_overlap_count': 0,
        })
        aggregate_rows.append(row)

threshold_metrics_agg = pd.DataFrame(aggregate_rows)
threshold_metrics = pd.concat([threshold_metrics_fold, threshold_metrics_agg], ignore_index=True)
metrics = pd.concat([
    metrics_fold,
    threshold_metrics_agg.loc[threshold_metrics_agg['threshold'].eq(DEFAULT_THRESHOLD)].copy()
], ignore_index=True)

predictions.to_csv(OUT_PREDICTIONS, index=False, encoding='utf-8-sig')
metrics.to_csv(OUT_METRICS, index=False, encoding='utf-8-sig')
threshold_metrics.to_csv(OUT_THRESHOLD_METRICS, index=False, encoding='utf-8-sig')

threshold_metrics_agg.sort_values(['dataset', 'threshold', 'balanced_accuracy'], ascending=[True, True, False]).head(20)

,threshold,rows,accuracy,balanced_accuracy,roc_auc,precision_event,recall_event,f1_event,normal_fpr,tn,...,feature_set,feature_count,model,fold,metric_scope,train_rows,test_rows,train_groups,test_groups,group_overlap_count
24,0.4,185,0.767568,0.637619,0.787238,0.863946,0.846667,0.855219,0.571429,15,...,expanded154,154,logistic_balanced,all,aggregate,NaN,185,NaN,NaN,0
9,0.4,185,0.729730,0.636190,0.723810,0.867647,0.786667,0.825175,0.514286,17,...,compact13,13,logistic_balanced,all,aggregate,NaN,185,NaN,NaN,0
12,0.4,185,0.789189,0.585238,0.694095,0.840491,0.913333,0.875399,0.742857,9,...,compact13,13,random_forest_balanced_depth3,all,aggregate,NaN,185,NaN,NaN,0
27,0.4,185,0.800000,0.580952,0.779810,0.838323,0.933333,0.883281,0.771429,8,...,expanded154,154,random_forest_balanced_depth3,all,aggregate,NaN,185,NaN,NaN,0
6,0.4,185,0.794595,0.555714,0.713714,0.829412,0.940000,0.881250,0.828571,6,...,compact13,13,hist_gradient_boosting,all,aggregate,NaN,185,NaN,NaN,0
21,0.4,185,0.772973,0.520476,0.744762,0.817647,0.926667,0.868750,0.885714,4,...,expanded154,154,hist_gradient_boosting,all,aggregate,NaN,185,NaN,NaN,0
18,0.4,185,0.783784,0.516190,0.756000,0.816092,0.946667,0.876543,0.914286,3,...,expanded154,154,extra_trees_balanced_depth3,all,aggregate,NaN,185,NaN,NaN,0
0,0.4,185,0.810811,0.500000,0.500000,0.810811,1.000000,0.895522,1.000000,0,...,compact13,13,dummy_most_frequent,all,aggregate,NaN,185,NaN,NaN,0
15,0.4,185,0.810811,0.500000,0.500000,0.810811,1.000000,0.895522,1.000000,0,...,expanded154,154,dummy_most_frequent,all,aggregate,NaN,185,NaN,NaN,0
3,0.4,185,0.794595,0.490000,0.715810,0.807692,0.980000,0.885542,1.000000,0,...,compact13,13,extra_trees_balanced_depth3,all,aggregate,NaN,185,NaN,NaN,0


## Decision Matrix

기본 의사결정은 threshold 0.5에서 수행하고, 0.4/0.6은 보조 판단 자료로 남긴다.

In [9]:
agg = threshold_metrics_agg.copy()
dummy_lookup = (
    agg.loc[agg['model'].eq('dummy_most_frequent'), ['dataset', 'feature_set', 'threshold', 'balanced_accuracy']]
    .rename(columns={'balanced_accuracy': 'dummy_balanced_accuracy'})
)
decision_matrix = agg.merge(dummy_lookup, on=['dataset', 'feature_set', 'threshold'], how='left')
decision_matrix['balanced_accuracy_lift_vs_dummy'] = decision_matrix['balanced_accuracy'] - decision_matrix['dummy_balanced_accuracy']
decision_matrix['passes_event_recall'] = decision_matrix['recall_event'] >= 0.80
decision_matrix['passes_normal_fpr'] = decision_matrix['normal_fpr'] <= 0.25
decision_matrix['passes_dummy_lift'] = decision_matrix['balanced_accuracy_lift_vs_dummy'] >= 0.15
decision_matrix['passes_group_overlap'] = decision_matrix['group_overlap_count'].eq(0)
decision_matrix['passes_gate_criteria'] = (
    decision_matrix['passes_event_recall']
    & decision_matrix['passes_normal_fpr']
    & decision_matrix['passes_dummy_lift']
    & decision_matrix['passes_group_overlap']
    & ~decision_matrix['model'].eq('dummy_most_frequent')
)

def decision_label(row: pd.Series) -> str:
    if not row['passes_gate_criteria']:
        return 'binary_gate_unstable'
    if row['dataset'] == 'pre_event_all':
        return 'pre_event_gate_possible'
    if row['dataset'] == 'low_overlap_hybrid':
        return 'hybrid_gate_only_possible'
    return 'binary_gate_unstable'

decision_matrix['decision_candidate'] = decision_matrix.apply(decision_label, axis=1)

# Overall conclusion is locked to default threshold 0.5.
default_non_dummy = decision_matrix.loc[
    decision_matrix['threshold'].eq(DEFAULT_THRESHOLD) & ~decision_matrix['model'].eq('dummy_most_frequent')
].copy()
if ((default_non_dummy['dataset'].eq('pre_event_all')) & default_non_dummy['passes_gate_criteria']).any():
    overall_decision = 'pre_event_gate_possible'
elif ((default_non_dummy['dataset'].eq('low_overlap_hybrid')) & default_non_dummy['passes_gate_criteria']).any():
    overall_decision = 'hybrid_gate_only_possible'
else:
    overall_decision = 'binary_gate_unstable'

decision_matrix['overall_decision_at_t0_5'] = overall_decision

decision_matrix.to_csv(OUT_DECISION_MATRIX, index=False, encoding='utf-8-sig')
print('overall_decision_at_t0_5:', overall_decision)
decision_matrix.sort_values(['threshold', 'passes_gate_criteria', 'balanced_accuracy'], ascending=[True, False, False]).head(20)

overall_decision_at_t0_5: binary_gate_unstable


,threshold,rows,accuracy,balanced_accuracy,roc_auc,precision_event,recall_event,f1_event,normal_fpr,tn,...,group_overlap_count,dummy_balanced_accuracy,balanced_accuracy_lift_vs_dummy,passes_event_recall,passes_normal_fpr,passes_dummy_lift,passes_group_overlap,passes_gate_criteria,decision_candidate,overall_decision_at_t0_5
39,0.4,181,0.712707,0.648141,0.700196,0.873016,0.753425,0.808824,0.457143,19,...,0,0.5,0.148141,False,False,False,True,False,binary_gate_unstable,binary_gate_unstable
24,0.4,185,0.767568,0.637619,0.787238,0.863946,0.846667,0.855219,0.571429,15,...,0,0.5,0.137619,True,False,False,True,False,binary_gate_unstable,binary_gate_unstable
9,0.4,185,0.729730,0.636190,0.723810,0.867647,0.786667,0.825175,0.514286,17,...,0,0.5,0.136190,False,False,False,True,False,binary_gate_unstable,binary_gate_unstable
54,0.4,181,0.734807,0.618395,0.699022,0.855072,0.808219,0.830986,0.571429,15,...,0,0.5,0.118395,True,False,False,True,False,binary_gate_unstable,binary_gate_unstable
57,0.4,181,0.817680,0.604599,0.793346,0.842424,0.952055,0.893891,0.742857,9,...,0,0.5,0.104599,True,False,False,True,False,binary_gate_unstable,binary_gate_unstable
42,0.4,181,0.790055,0.598337,0.711155,0.841772,0.910959,0.875000,0.714286,10,...,0,0.5,0.098337,True,False,False,True,False,binary_gate_unstable,binary_gate_unstable
36,0.4,181,0.806630,0.597750,0.718982,0.840491,0.938356,0.886731,0.742857,9,...,0,0.5,0.097750,True,False,False,True,False,binary_gate_unstable,binary_gate_unstable
12,0.4,185,0.789189,0.585238,0.694095,0.840491,0.913333,0.875399,0.742857,9,...,0,0.5,0.085238,True,False,False,True,False,binary_gate_unstable,binary_gate_unstable
27,0.4,185,0.800000,0.580952,0.779810,0.838323,0.933333,0.883281,0.771429,8,...,0,0.5,0.080952,True,False,False,True,False,binary_gate_unstable,binary_gate_unstable
48,0.4,181,0.801105,0.561742,0.781213,0.827381,0.952055,0.885350,0.828571,6,...,0,0.5,0.061742,True,False,False,True,False,binary_gate_unstable,binary_gate_unstable


## Validation Checks

In [10]:
# Required dataset checks
assert int(dataset_summary.loc[dataset_summary['dataset'].eq('pre_event_all'), 'normal_rows'].iloc[0]) == 35
assert int(dataset_summary.loc[dataset_summary['dataset'].eq('low_overlap_hybrid'), 'normal_rows'].iloc[0]) == 35
assert set(DATASETS['pre_event_all']['final_class'].unique()) == {'normal', 'fault', 'task', 'activity'}
assert set(DATASETS['low_overlap_hybrid']['final_class'].unique()) == {'normal', 'fault', 'task', 'activity'}

# Feature count checks
assert len(FEATURE_COLUMNS['compact13']) == 13
assert len(FEATURE_COLUMNS['expanded154']) == 154

# Group overlap check
assert threshold_metrics['group_overlap_count'].fillna(0).max() == 0

# M1-only string check for new outputs. Avoid writing non-target manufacturer identifiers.
new_output_paths = [
    OUT_DATASET_SUMMARY,
    OUT_METRICS,
    OUT_PREDICTIONS,
    OUT_THRESHOLD_METRICS,
    OUT_DECISION_MATRIX,
]
for path in new_output_paths:
    text = path.read_text(encoding='utf-8-sig', errors='ignore')
    lowered = text.lower()
    assert ('manufacturer' + '_2') not in lowered
    assert ('manufacturer' + ' ' + '2') not in lowered

print('validation checks passed')

validation checks passed


## Results

In [11]:
best_default = (
    decision_matrix.loc[
        decision_matrix['threshold'].eq(DEFAULT_THRESHOLD) & ~decision_matrix['model'].eq('dummy_most_frequent')
    ]
    .sort_values(['passes_gate_criteria', 'balanced_accuracy', 'recall_event'], ascending=[False, False, False])
    .head(10)
)
best_default[[
    'dataset', 'feature_set', 'model', 'threshold', 'rows', 'balanced_accuracy',
    'balanced_accuracy_lift_vs_dummy', 'recall_event', 'normal_fpr', 'precision_event',
    'f1_event', 'fp', 'fn', 'passes_gate_criteria', 'decision_candidate'
]]

,dataset,feature_set,model,threshold,rows,balanced_accuracy,balanced_accuracy_lift_vs_dummy,recall_event,normal_fpr,precision_event,f1_event,fp,fn,passes_gate_criteria,decision_candidate
58,pre_event_all,expanded154,random_forest_balanced_depth3,0.5,181,0.725832,0.225832,0.794521,0.342857,0.906250,0.846715,12,30,False,binary_gate_unstable
34,pre_event_all,compact13,extra_trees_balanced_depth3,0.5,181,0.707632,0.207632,0.643836,0.228571,0.921569,0.758065,8,52,False,binary_gate_unstable
19,low_overlap_hybrid,expanded154,extra_trees_balanced_depth3,0.5,185,0.703810,0.203810,0.493333,0.085714,0.961039,0.651982,3,76,False,binary_gate_unstable
25,low_overlap_hybrid,expanded154,logistic_balanced,0.5,185,0.699048,0.199048,0.826667,0.428571,0.892086,0.858131,15,26,False,binary_gate_unstable
49,pre_event_all,expanded154,extra_trees_balanced_depth3,0.5,181,0.696869,0.196869,0.479452,0.085714,0.958904,0.639269,3,76,False,binary_gate_unstable
28,low_overlap_hybrid,expanded154,random_forest_balanced_depth3,0.5,185,0.685714,0.185714,0.800000,0.428571,0.888889,0.842105,15,30,False,binary_gate_unstable
10,low_overlap_hybrid,compact13,logistic_balanced,0.5,185,0.675238,0.175238,0.693333,0.342857,0.896552,0.781955,12,46,False,binary_gate_unstable
4,low_overlap_hybrid,compact13,extra_trees_balanced_depth3,0.5,185,0.661905,0.161905,0.666667,0.342857,0.892857,0.763359,12,50,False,binary_gate_unstable
40,pre_event_all,compact13,logistic_balanced,0.5,181,0.650489,0.150489,0.643836,0.342857,0.886792,0.746032,12,52,False,binary_gate_unstable
43,pre_event_all,compact13,random_forest_balanced_depth3,0.5,181,0.633855,0.133855,0.753425,0.485714,0.866142,0.805861,17,36,False,binary_gate_unstable


In [12]:
threshold_view = (
    decision_matrix.loc[~decision_matrix['model'].eq('dummy_most_frequent')]
    .sort_values(['threshold', 'balanced_accuracy'], ascending=[True, False])
    .groupby('threshold')
    .head(5)
)
threshold_view[[
    'threshold', 'dataset', 'feature_set', 'model', 'balanced_accuracy',
    'balanced_accuracy_lift_vs_dummy', 'recall_event', 'normal_fpr', 'fp', 'fn',
    'passes_gate_criteria'
]]

,threshold,dataset,feature_set,model,balanced_accuracy,balanced_accuracy_lift_vs_dummy,recall_event,normal_fpr,fp,fn,passes_gate_criteria
39,0.4,pre_event_all,compact13,logistic_balanced,0.648141,0.148141,0.753425,0.457143,16,36,False
24,0.4,low_overlap_hybrid,expanded154,logistic_balanced,0.637619,0.137619,0.846667,0.571429,20,23,False
9,0.4,low_overlap_hybrid,compact13,logistic_balanced,0.636190,0.136190,0.786667,0.514286,18,32,False
54,0.4,pre_event_all,expanded154,logistic_balanced,0.618395,0.118395,0.808219,0.571429,20,28,False
57,0.4,pre_event_all,expanded154,random_forest_balanced_depth3,0.604599,0.104599,0.952055,0.742857,26,7,False
58,0.5,pre_event_all,expanded154,random_forest_balanced_depth3,0.725832,0.225832,0.794521,0.342857,12,30,False
34,0.5,pre_event_all,compact13,extra_trees_balanced_depth3,0.707632,0.207632,0.643836,0.228571,8,52,False
19,0.5,low_overlap_hybrid,expanded154,extra_trees_balanced_depth3,0.703810,0.203810,0.493333,0.085714,3,76,False
25,0.5,low_overlap_hybrid,expanded154,logistic_balanced,0.699048,0.199048,0.826667,0.428571,15,26,False
49,0.5,pre_event_all,expanded154,extra_trees_balanced_depth3,0.696869,0.196869,0.479452,0.085714,3,76,False


## Write Report

In [13]:
def fmt_float(value, digits=4):
    if pd.isna(value):
        return 'NA'
    return f'{float(value):.{digits}f}'


def md_table(df: pd.DataFrame) -> str:
    table = df.copy()
    for col in table.columns:
        table[col] = table[col].map(lambda x: '' if pd.isna(x) else str(x))
    header = '| ' + ' | '.join(table.columns) + ' |'
    sep = '| ' + ' | '.join(['---'] * len(table.columns)) + ' |'
    body = ['| ' + ' | '.join(row) + ' |' for row in table.astype(str).values.tolist()]
    return '\n'.join([header, sep] + body)

best_rows = best_default.copy()
best_line = best_rows.iloc[0]

passes_default = default_non_dummy.loc[default_non_dummy['passes_gate_criteria']].copy()
threshold04_passes = decision_matrix.loc[
    decision_matrix['threshold'].eq(0.4) & decision_matrix['passes_gate_criteria'] & ~decision_matrix['model'].eq('dummy_most_frequent')
].copy()

if overall_decision == 'pre_event_gate_possible':
    conclusion_text = 'threshold 0.5 기준에서 `pre_event_all` gate가 채택 조건을 만족한다. 다음 단계는 event 내부 `fault/task/activity` 3분류 검증이다.'
elif overall_decision == 'hybrid_gate_only_possible':
    conclusion_text = 'threshold 0.5 기준에서 `pre_event_all`은 부족하지만 `low_overlap_hybrid`는 채택 조건을 만족한다. 다음 단계는 hybrid window 정책을 기준으로 event 내부 3분류를 검증한다.'
else:
    conclusion_text = 'threshold 0.5 기준에서 채택 조건을 만족한 조합이 없다. 4분류로 바로 확정하기보다 window policy 또는 normal/event 후보 선별을 먼저 재검토해야 한다.'

lines = []
lines.append('# M1 4class Binary Gate 검증 보고서')
lines.append('')
lines.append('## 결론')
lines.append(f'- 최종 판단: `{overall_decision}`')
lines.append(f'- 해석: {conclusion_text}')
lines.append(f'- 최고 기본 threshold 조합: `{best_line["dataset"]}` / `{best_line["feature_set"]}` / `{best_line["model"]}`')
lines.append(f'- 최고 기본 threshold 성능: balanced accuracy `{fmt_float(best_line["balanced_accuracy"])}`, event recall `{fmt_float(best_line["recall_event"])}`, normal FPR `{fmt_float(best_line["normal_fpr"])}`')
lines.append('')
lines.append('## Dataset 구성')
lines.append(md_table(dataset_summary))
lines.append('')
lines.append('## Threshold 0.5 주요 결과')
show_cols = ['dataset', 'feature_set', 'model', 'balanced_accuracy', 'balanced_accuracy_lift_vs_dummy', 'recall_event', 'normal_fpr', 'precision_event', 'f1_event', 'fp', 'fn', 'passes_gate_criteria']
lines.append(md_table(best_default[show_cols]))
lines.append('')
lines.append('## Threshold 참고 결과')
lines.append('- 0.5를 기본 판단 기준으로 유지했다.')
lines.append('- 0.4는 event recall 보완 가능성 확인용, 0.6은 normal 오탐 억제 가능성 확인용이다.')
lines.append(md_table(threshold_view[['threshold', 'dataset', 'feature_set', 'model', 'balanced_accuracy', 'recall_event', 'normal_fpr', 'fp', 'fn', 'passes_gate_criteria']]))
lines.append('')
lines.append('## 검증 사항')
lines.append('- 모든 입력은 16번 M1 taxonomy 산출물에서만 사용했다.')
lines.append('- normal 35건은 그대로 유지했다.')
lines.append('- 학습 입력에는 metadata, 날짜, event id, substation, coverage를 넣지 않았다.')
lines.append('- `compact13`은 13개, `expanded154`는 154개 feature로 분리했다.')
lines.append('- group CV train/test substation overlap은 모든 fold에서 0이다.')
lines.append('- 비대상 제조사 문자열/경로/계산 결과는 새 산출물에 포함하지 않았다.')
lines.append('')
lines.append('## 다음 단계')
if overall_decision in {'pre_event_gate_possible', 'hybrid_gate_only_possible'}:
    lines.append('- `normal vs event` gate 기준을 고정 후보로 두고, event 내부 `fault/task/activity` 3분류를 검증한다.')
else:
    lines.append('- 4분류 모델 확정 전에 event window policy와 normal/event 후보 선별 기준을 재검토한다.')
    if not threshold04_passes.empty:
        lines.append('- 참고로 threshold 0.4에서는 일부 조합이 조건을 만족하므로, 운영 목적이 event recall 우선인지 normal 오탐 억제 우선인지 별도 판단이 필요하다.')

OUT_REPORT.write_text('\n'.join(lines) + '\n', encoding='utf-8')
print(OUT_REPORT)

C:\3rd_Project\HeatGridAgent\07_데이터산출물\17_M1_4class_binary_gate_검증_보고서.md


## Takeaways

보고서와 CSV 산출물을 기준으로 `normal vs event` gate 채택 여부를 판단한다.